# RoboCoach development notebook — historical archive

A cleaned copy of the original working notebook supplied by the author. Outputs, execution counts, transient cell metadata, and Colab account metadata have been removed. Source cells are preserved.

**Do not Run all.** This is a development record: it includes failed attempts, cells that depend on temporary runtime variables, old Drive paths, two-epoch smoke tests, and annotation repair experiments. Use [train_obb.ipynb](./train_obb.ipynb) for a configurable training workflow.

The historical repair cells removed one edge annotation and shifted 12 edge boxes in the Toggle validation data. Those edits are not a recommended automatic labeling policy; review edge annotations visually. Compare models on the same reviewed validation labels.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!unzip -q -o "/content/drive/MyDrive/Toggle2.zip" -d /content/toggle_datasets && find /content/toggle_datasets -name data.yaml

In [ ]:
%pip install -q ultralytics

In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO

root = Path("/content/toggle_datasets")
config = {"names": {0: "toggle"}}

for split, name in [("train", "Toggle1"), ("val", "Toggle2")]:
    image_dir = root / name / "images"
    images = sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    )
    labels = [
        (root / name / "labels" / p.relative_to(image_dir)).with_suffix(".txt")
        for p in images
    ]
    labeled = sum(p.is_file() and bool(p.read_text().strip()) for p in labels)
    print(f"{name}: {len(images)} images, {labeled} with boxes")
    if not labeled:
        raise ValueError(f"No images with matching labels found in {name}")

    manifest = root / f"{split}_images.txt"
    manifest.write_text("\n".join(str(p) for p in images) + "\n")
    config[split] = str(manifest)

config_path = root / "train_toggles.yaml"
config_path.write_text(yaml.safe_dump(config))

model = YOLO("yolo26n-obb.pt")
model.train(data=str(config_path), epochs=2, imgsz=640, device="cpu")

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

root = Path("/content/toggle_datasets")
image_dir = root / "Toggle1" / "images"
manifest = root / "train_images.txt"

for line in manifest.read_text().splitlines():
    image = Path(line)
    label = (root / "Toggle1" / "labels" / image.relative_to(image_dir)).with_suffix(".txt")
    if not label.exists() or not label.read_text().strip():
        print(image.name)
        with Image.open(image) as frame:
            frame.thumbnail((720, 480))
            display(frame)

In [ ]:
from collections import defaultdict
from pathlib import Path
import math
import re
import xml.etree.ElementTree as ET

from PIL import Image
import yaml

root = Path("/content/toggle_datasets")
xml = ET.parse(destination / "annotations.xml").getroot()
image_root = destination / "images"

boxes_by_frame = defaultdict(list)
for track in xml.findall("track"):
    if track.get("label", "").casefold() != "toggle":
        continue
    for box in track.findall("box"):
        if box.get("outside") == "0":
            boxes_by_frame[int(box.attrib["frame"])].append(box)

train_images = []
box_count = 0
skipped_images = []
matched_frames = set()

for image in sorted(image_root.rglob("*")):
    match = re.fullmatch(r"frame_(\d+)\.png", image.name, flags=re.IGNORECASE)
    if not match or not image.is_file():
        continue

    frame_number = int(match.group(1))
    boxes = boxes_by_frame.get(frame_number)
    if not boxes:
        skipped_images.append(image.name)
        continue

    with Image.open(image) as frame:
        width, height = frame.size

    rows = []
    for box in boxes:
        x1, y1, x2, y2 = (
            float(box.attrib[key]) for key in ("xtl", "ytl", "xbr", "ybr")
        )
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        angle = math.radians(float(box.get("rotation", "0")))
        c, s = math.cos(angle), math.sin(angle)
        corners = []

        for x, y in [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]:
            dx, dy = x - cx, y - cy
            u = (cx + c * dx - s * dy) / width
            v = (cy + s * dx + c * dy) / height
            if not (0 <= u <= 1 and 0 <= v <= 1):
                raise ValueError(f"A rotated box extends outside {image.name}.")
            corners.extend([u, v])

        rows.append("0 " + " ".join(f"{value:.8f}" for value in corners))

    label = (destination / "labels" / image.relative_to(image_root)).with_suffix(".txt")
    label.parent.mkdir(parents=True, exist_ok=True)
    label.write_text("\n".join(rows) + "\n")
    train_images.append(image)
    matched_frames.add(frame_number)
    box_count += len(rows)

unmatched = sorted(boxes_by_frame.keys() - matched_frames)
if unmatched:
    raise ValueError(f"Visible boxes have no matching PNG for frames {unmatched}")
if not train_images:
    raise ValueError("No labeled training images were found.")

val_images = sorted(
    image for image in (root / "Toggle2" / "images").rglob("*")
    if image.is_file() and image.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
if not val_images:
    raise ValueError("No validation images found in Toggle2.")

config = {"names": {0: "toggle"}}
for split, images in [("train", train_images), ("val", val_images)]:
    manifest = root / f"{split}_images.txt"
    manifest.write_text("\n".join(str(image) for image in images) + "\n")
    config[split] = str(manifest)

config_path = root / "train_toggles.yaml"
config_path.write_text(yaml.safe_dump(config))

print(f"Training: {len(train_images)} labeled images, {box_count} boxes")
print(f"Validation: {len(val_images)} images")
print("Skipped images without explicit visible boxes:", skipped_images)
print(f"Ready to train with {config_path}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-obb.pt")
results = model.train(
    data="/content/toggle_datasets/train_toggles.yaml",
    epochs=2,
    imgsz=640,
    device="cpu",
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-obb.pt")
results = model.train(
    data="/content/toggle_datasets/train_toggles.yaml",
    epochs=50,
    imgsz=640,
    device="cpu",
    project="/content/drive/MyDrive/RoboCoach",
    name="toggle-obb",
)
print(results.save_dir)

In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/drive/MyDrive/RoboCoach/toggle-obb/val_batch0_pred.jpg"))

In [ ]:
from IPython.display import Image, display

run = "/content/drive/MyDrive/RoboCoach/toggle-obb"
batch = 0  # Change to 1 or 2 to review more frames.

print("Your CVAT labels")
display(Image(filename=f"{run}/val_batch{batch}_labels.jpg"))

print("YOLO predictions")
display(Image(filename=f"{run}/val_batch{batch}_pred.jpg"))

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

matches = [
    p for p in Path("/content/drive/MyDrive").rglob("*.zip")
    if "q4" in p.name.lower()
]

for p in matches:
    print(p)

if not matches:
    print("No Q4 ZIP found. Check that the upload finished in the same Google account.")

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile
from collections import Counter

archive = Path("/content/drive/MyDrive/RoboCoach/Q4 Toggle Training.zip")
q4_folder = Path(mkdtemp(prefix="q4-toggles-", dir="/content"))

with ZipFile(archive) as zip_file:
    zip_file.extractall(q4_folder)

files = sorted(p for p in q4_folder.rglob("*") if p.is_file())
print("File types:", Counter(p.suffix.lower() for p in files))

for p in files[:15]:
    print(p.relative_to(q4_folder))

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile
from collections import defaultdict
from PIL import Image
import math
import re
import shutil
import xml.etree.ElementTree as ET
import yaml

drive = Path("/content/drive/MyDrive")
work = Path(mkdtemp(prefix="toggle-setup-", dir="/content"))

def extract(archive, folder):
    with ZipFile(archive) as z:
        z.extractall(folder)
    return folder

original = extract(drive / "Toggle2.zip", work / "original")
fixed = extract(drive / "Toggle2.1.zip", work / "fixed")
q4 = extract(drive / "RoboCoach/Q4 Toggle Training.zip", work / "q4")

# Convert explicitly labeled rotated rectangles from the corrected CVAT XML.
xml_files = list(fixed.rglob("annotations.xml"))
if len(xml_files) != 1:
    raise ValueError("Expected one annotations.xml in Toggle2.1.zip.")

fixed_root = xml_files[0].parent
xml = ET.parse(xml_files[0]).getroot()
boxes = defaultdict(list)

for track in xml.findall("track"):
    if track.get("label", "").casefold() == "toggle":
        for box in track.findall("box"):
            if box.get("outside") == "0":
                boxes[int(box.attrib["frame"])].append(box)

for image in (fixed_root / "images").rglob("*"):
    match = re.fullmatch(r"frame_(\d+)\.png", image.name, re.IGNORECASE)
    if not match or not image.is_file():
        continue

    frame_boxes = boxes.get(int(match.group(1)), [])
    if not frame_boxes:
        continue

    with Image.open(image) as picture:
        width, height = picture.size

    rows = []
    for box in frame_boxes:
        x1, y1, x2, y2 = [
            float(box.attrib[k]) for k in ("xtl", "ytl", "xbr", "ybr")
        ]
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        angle = math.radians(float(box.get("rotation", 0)))
        c, s = math.cos(angle), math.sin(angle)
        coordinates = []

        for x, y in [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]:
            dx, dy = x - cx, y - cy
            coordinates.extend([
                (cx + c * dx - s * dy) / width,
                (cy + s * dx + c * dy) / height,
            ])

        if not all(0 <= value <= 1 for value in coordinates):
            raise ValueError(f"Box outside image boundaries: {image.name}")

        rows.append("0 " + " ".join(f"{v:.8f}" for v in coordinates))

    label = (
        fixed_root / "labels" /
        image.relative_to(fixed_root / "images")
    ).with_suffix(".txt")
    label.parent.mkdir(parents=True, exist_ok=True)
    label.write_text("\n".join(rows) + "\n")

# Locate the validation video and Q4 export.
validation_roots = [
    p for p in original.rglob("Toggle2")
    if p.is_dir() and (p / "images").is_dir()
]
if len(validation_roots) != 1:
    raise ValueError("Could not uniquely locate Toggle2/images in the original ZIP.")

q4_configs = list(q4.rglob("data.yaml"))
if len(q4_configs) != 1:
    raise ValueError("Expected one data.yaml in the Q4 ZIP.")

# Create a fresh, persistent dataset. Original exports stay intact.
prepared = Path(mkdtemp(
    prefix="prepared-toggles-",
    dir=str(drive / "RoboCoach"),
))

def copy_dataset(source, split, prefix):
    image_root = source / "images"
    count = 0

    for image in sorted(image_root.rglob("*")):
        if not image.is_file() or image.suffix.lower() not in {
            ".png", ".jpg", ".jpeg", ".webp"
        }:
            continue

        label = (
            source / "labels" / image.relative_to(image_root)
        ).with_suffix(".txt")

        # Only include frames explicitly labeled with Toggle boxes.
        if not label.exists() or not label.read_text().strip():
            continue

        for line in label.read_text().splitlines():
            values = line.split()
            if (
                len(values) != 9
                or values[0] != "0"
                or not all(0 <= float(v) <= 1 for v in values[1:])
            ):
                raise ValueError(f"Invalid single-class OBB label: {label}")

        name = f"{prefix}_{count:05d}"
        image_target = prepared / "images" / split / (name + image.suffix.lower())
        label_target = prepared / "labels" / split / (name + ".txt")
        image_target.parent.mkdir(parents=True, exist_ok=True)
        label_target.parent.mkdir(parents=True, exist_ok=True)

        shutil.copyfile(image, image_target)
        shutil.copyfile(label, label_target)
        count += 1

    if not count:
        raise ValueError(f"No matching labeled images found in {source}")

    return count

old_count = copy_dataset(fixed_root, "train", "original")
q4_count = copy_dataset(q4_configs[0].parent, "train", "q4")
val_count = copy_dataset(validation_roots[0], "val", "validation")

config_path = prepared / "data.yaml"
config_path.write_text(yaml.safe_dump({
    "path": str(prepared),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "toggle"},
}))

print(f"Training: {old_count} original + {q4_count} Q4 images")
print(f"Validation: {val_count} labeled images")
print("Saved dataset config:", config_path)

In [ ]:
label = q4_configs[0].parent / "labels/Train/frame_003120.txt"

backup = label.with_suffix(".txt.bak")
if not backup.exists():
    backup.write_text(label.read_text())

lines = label.read_text().splitlines()
label.write_text(
    "\n".join(line for line in lines if "-0.004626" not in line) + "\n"
)

q4_count = copy_dataset(q4_configs[0].parent, "train", "q4")
val_count = copy_dataset(validation_roots[0], "val", "validation")

config_path = prepared / "data.yaml"
config_path.write_text(yaml.safe_dump({
    "path": str(prepared),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "toggle"},
}))

print(f"Training: {old_count} original + {q4_count} Q4 images")
print(f"Validation: {val_count} images")
print("Ready:", config_path)

In [ ]:
source = validation_roots[0]
changed = 0

for label in sorted((source / "labels").rglob("*.txt")):
    original_text = label.read_text()
    output = []

    for line in original_text.splitlines():
        values = line.split()
        if len(values) != 9 or values[0] != "0":
            raise ValueError(f"Unexpected label format: {label}")

        coordinates = [float(v) for v in values[1:]]
        xs, ys = coordinates[0::2], coordinates[1::2]

        if max(xs) - min(xs) > 1 or max(ys) - min(ys) > 1:
            raise ValueError(f"Box is too large to fit inside the image: {label}")

        dx = -min(xs) if min(xs) < 0 else min(0, 1 - max(xs))
        dy = -min(ys) if min(ys) < 0 else min(0, 1 - max(ys))

        if dx or dy:
            coordinates = [
                value + (dx if i % 2 == 0 else dy)
                for i, value in enumerate(coordinates)
            ]
            output.append("0 " + " ".join(f"{v:.8f}" for v in coordinates))
            changed += 1
        else:
            output.append(line)

    updated = "\n".join(output) + "\n"
    if updated != original_text:
        backup = label.with_suffix(".txt.bak")
        if not backup.exists():
            backup.write_text(original_text)
        label.write_text(updated)

val_count = copy_dataset(source, "val", "validation")

config_path = prepared / "data.yaml"
config_path.write_text(yaml.safe_dump({
    "path": str(prepared),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "toggle"},
}))

print(f"Adjusted {changed} boxes")
print(f"Training: {old_count} original + {q4_count} Q4 images")
print(f"Validation: {val_count} images")
print("Ready:", config_path)

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/RoboCoach/toggle-obb/weights/best.pt"
)

results = model.train(
    data="/content/drive/MyDrive/RoboCoach/prepared-toggles-3d8fykqf/data.yaml",
    epochs=50,
    imgsz=640,
    device="cpu",
    project="/content/drive/MyDrive/RoboCoach",
    name="toggle-obb-with-q4",
    exist_ok=False,
)

In [ ]:
%pip install -q ultralytics

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/RoboCoach")

for folder in sorted(root.glob("toggle-obb-with-online*")):
    print("\nRun:", folder.name)
    print("Checkpoint exists:", (folder / "weights/last.pt").exists())

    results = folder / "results.csv"
    if results.exists():
        lines = results.read_text().splitlines()
        print("Latest saved result:", lines[-1])

In [ ]:
from IPython.display import Image, display

display(Image(
    filename="/content/drive/MyDrive/RoboCoach/toggle-obb-with-online/val_batch0_pred.jpg"
))

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/RoboCoach/"
    "toggle-obb-with-online/weights/best.pt"
)

onnx_path = model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    dynamic=False,
    simplify=True,
    device="cpu",
)

print("Saved:", onnx_path)

In [ ]:
%pip install -q ultralytics

from pathlib import Path
from tempfile import mkdtemp
import shutil
import yaml
from ultralytics import YOLO

source = Path(
    "/content/drive/MyDrive/RoboCoach/pins-cups-first50-isu4riuc"
)
images = sorted((source / "images/train").glob("*.png"))
assert len(images) == 50, f"Expected 50 images, found {len(images)}"

dataset = Path(mkdtemp(
    prefix="pins-cups-split-",
    dir="/content/drive/MyDrive/RoboCoach"
))

for split, selected in [
    ("train", images[:40]),
    ("val", images[40:]),
]:
    (dataset / "images" / split).mkdir(parents=True)
    (dataset / "labels" / split).mkdir(parents=True)

    for image in selected:
        label = source / "labels/train" / f"{image.stem}.txt"
        shutil.copyfile(image, dataset / "images" / split / image.name)
        shutil.copyfile(label, dataset / "labels" / split / label.name)

config = dataset / "data.yaml"
config.write_text(yaml.safe_dump({
    "path": str(dataset),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "Toggle",
        1: "red_pin",
        2: "blue_pin",
        3: "yellow_pin",
        4: "cup",
    },
}))

print("Dataset:", dataset)
print("Training: 40 images | Validation: 10 images")

model = YOLO(
    "/content/drive/MyDrive/RoboCoach/"
    "toggle-obb-with-online/weights/best.pt"
)

results = model.train(
    data=str(config),
    epochs=50,
    imgsz=640,
    device="cpu",
    project="/content/drive/MyDrive/RoboCoach",
    name="pins-cups-obb",
    exist_ok=False,
)

In [ ]:
from IPython.display import Image, display

folder = "/content/drive/MyDrive/RoboCoach/pins-cups-obb"

print("YOUR LABELS")
display(Image(filename=f"{folder}/val_batch0_labels.jpg"))

print("MODEL PREDICTIONS")
display(Image(filename=f"{folder}/val_batch0_pred.jpg"))

In [ ]:
from pathlib import Path
from ultralytics import YOLO

run = Path("/content/drive/MyDrive/RoboCoach/pins-cups-obb")

# Find the exact validation images used for this run.
import yaml
settings = yaml.safe_load((run / "args.yaml").read_text())
config = yaml.safe_load(Path(settings["data"]).read_text())
validation = Path(config["path"]) / config["val"]

model = YOLO(str(run / "weights/best.pt"))

for result in model.predict(
    source=str(validation),
    conf=0.25,
    save=True,
    show_labels=False,
    show_conf=False,
    line_width=1,
    stream=True,
    project="/content/drive/MyDrive/RoboCoach",
    name="pins-cups-clean-preview",
):
    pass